In [1]:
import torch
print(torch.__version__, torch.version.cuda)
print(torch.cuda.is_available())

2.14.0+cu126 12.6
True


In [ ]:
import glob
from datasets import load_dataset
from collections import Counter
import numpy as np

def load_full(glob_path):
    files = sorted(glob.glob(glob_path, recursive=True))
    return load_dataset("parquet", data_files=files, split="train")

print("Loading full train set...")
train_ds = load_full("data_full_raw/train/data/*.parquet")
print(f"Total rows: {len(train_ds)}")


In [ ]:
langs = Counter(train_ds["language"])
print("=== Language distribution ===")
for lang, count in langs.most_common():
    print(f"  {lang}: {count} ({100*count/len(train_ds):.2f}%)")

In [ ]:
labels = train_ds["endpoint_bool"]
pos = sum(1 for v in labels if v)
print("=== Label balance ===")
print(f"  True (complete): {pos} ({100*pos/len(labels):.2f}%)")
print(f"  False (incomplete): {len(labels)-pos} ({100*(len(labels)-pos)/len(labels):.2f}%)")

In [ ]:
synthetic = train_ds["synthetic"]
synth_pct = 100 * sum(1 for v in synthetic if v) / len(synthetic)
print("=== Synthetic ratio ===")
print(f"  Overall: {synth_pct:.2f}% synthetic")

dataset_col = train_ds["dataset"]
print("\n=== Source subset distribution ===")
for name, count in Counter(dataset_col).most_common():
    print(f"  {name}: {count} ({100*count/len(dataset_col):.2f}%)")

In [ ]:
real_mask = [not v for v in synthetic]
real_langs = Counter([l for l, r in zip(train_ds["language"], real_mask) if r])
print("=== Real (non-synthetic) rows by language ===")
print(f"  Total real rows: {sum(real_langs.values())} / {len(train_ds)} "
      f"({100*sum(real_langs.values())/len(train_ds):.2f}%)")
for lang, count in real_langs.most_common():
    print(f"  {lang}: {count}")

hindi_real = real_langs.get("hin", 0)
print(f"\n  Hindi real rows: {hindi_real}")

In [ ]:
midfiller = train_ds["midfiller"]
endfiller = train_ds["endfiller"]
subset_null_stats = {}
for ds_name, mf, ef in zip(dataset_col, midfiller, endfiller):
    subset_null_stats.setdefault(ds_name, {"total": 0, "has_filler_info": 0})
    subset_null_stats[ds_name]["total"] += 1
    if mf is not None or ef is not None:
        subset_null_stats[ds_name]["has_filler_info"] += 1

print("=== midfiller/endfiller availability by source subset ===")
for name, stats in subset_null_stats.items():
    pct = 100 * stats["has_filler_info"] / stats["total"]
    print(f"  {name}: {stats['has_filler_info']}/{stats['total']} have filler annotations ({pct:.1f}%)")

# Stage 2 data

In [ ]:
from datasets import load_dataset
from collections import Counter
import glob

hindi_files = sorted(glob.glob("data_diarbench_raw/Hindi/**/*.parquet", recursive=True))
print(f"Found {len(hindi_files)} Hindi parquet files")

ds = load_dataset("parquet", data_files=hindi_files, split="train")
print(f"Total Hindi samples: {len(ds)}")
print(f"Columns: {ds.column_names}\n")
print(f"dataset_type distribution: {Counter(ds['dataset_type'])}\n")

In [ ]:
from datasets import load_dataset
from collections import Counter
import glob

hindi_files = sorted(glob.glob("data_diarbench_raw/Hindi/**/*.parquet", recursive=True))
print(f"Found {len(hindi_files)} Hindi parquet files\n")

ds = load_dataset("parquet", data_files=hindi_files, split="train")
print(f"Total Hindi samples: {len(ds)}")
print(f"Columns: {ds.column_names}\n")

print(f"dataset_type distribution: {Counter(ds['dataset_type'])}\n")

durations = ds['duration_seconds']
n_segments = ds['num_segments']
n_speakers = ds['num_speakers']
print(f"duration_seconds: min={min(durations):.1f} max={max(durations):.1f} avg={sum(durations)/len(durations):.1f}")
print(f"num_segments: min={min(n_segments)} max={max(n_segments)} avg={sum(n_segments)/len(n_segments):.1f}")
print(f"num_speakers: min={min(n_speakers)} max={max(n_speakers)} avg={sum(n_speakers)/len(n_speakers):.1f}")

sample = ds[0]
print(f"\n=== Sample 0 ===")
print(f"sample_id: {sample['sample_id']}, duration: {sample['duration_seconds']:.1f}s, "
      f"speakers: {sample['num_speakers']}, segments: {sample['num_segments']}")

segs = sample['annotated_transcript']
print(f"\nFirst 10 segments:")
for seg in segs[:10]:
    dur = seg['end_time'] - seg['start_time']
    print(f"  [{seg['start_time']:.2f}-{seg['end_time']:.2f}] ({dur:.2f}s) {seg['speaker_id']}: {seg['transcript'][:60]}")

# --- Aggregate overlap / same-speaker / segment-length stats across ALL samples ---
total_transitions = 0
overlap_count = 0
same_speaker_count = 0
all_seg_durs = []
substantive_skip_tags = ("<unintelligible>", "<vocalization>", "<laughter>")

for s in ds:
    segs = s['annotated_transcript']
    for seg in segs:
        all_seg_durs.append(seg['end_time'] - seg['start_time'])
    for i in range(len(segs) - 1):
        cur, nxt = segs[i], segs[i+1]
        if cur['transcript'].strip() in substantive_skip_tags:
            continue  # don't count transitions FROM a noise segment
        total_transitions += 1
        if nxt['start_time'] < cur['end_time']:
            overlap_count += 1
        if cur['speaker_id'] == nxt['speaker_id']:
            same_speaker_count += 1

print(f"\n=== Aggregate stats across all {len(ds)} Hindi samples ===")
print(f"Total transitions (from substantive segments): {total_transitions}")
print(f"  Overlapping: {overlap_count} ({100*overlap_count/total_transitions:.1f}%)")
print(f"  Same-speaker: {same_speaker_count} ({100*same_speaker_count/total_transitions:.1f}%)")

long_segs = sum(1 for d in all_seg_durs if d > 8)
print(f"\nSegment durations: min={min(all_seg_durs):.2f} max={max(all_seg_durs):.2f} "
      f"avg={sum(all_seg_durs)/len(all_seg_durs):.2f}")
print(f"Total segments: {len(all_seg_durs)}, >8s (usable for mid-cut negatives): {long_segs}")

audio = sample['audio']
if hasattr(audio, 'get_all_samples'):
    samples = audio.get_all_samples()
    print(f"\naudio shape: {samples.data.shape}, sample_rate: {samples.sample_rate}")

In [ ]:
NOISE_TAGS = ("<unintelligible>", "<vocalization>", "<laughter>")
MIN_SUBSTANTIVE_DURATION = 1.0  # seconds - segments shorter than this treated as backchannel-like noise



def is_substantive(seg):
    text = seg['transcript'].strip()
    dur = seg['end_time'] - seg['start_time']
    if dur <= 0:
        return False  # invalid/corrupt annotation
    if text in NOISE_TAGS:
        return False
    if dur < MIN_SUBSTANTIVE_DURATION:
        return False
    return True

total_raw_segments = 0
invalid_duration_segments = 0
noise_tag_segments = 0
short_backchannel_segments = 0
substantive_segments = 0

total_transitions = 0
overlap_count = 0
same_speaker_count = 0
diff_speaker_clean_count = 0  # usable "complete" examples
same_speaker_clean_count = 0  # usable "incomplete" examples (no overlap)
same_speaker_overlap_count = 0  # usable "incomplete" examples (overlap, same speaker - strong signal)
diff_speaker_overlap_count = 0  # excluded - genuine ambiguous interruption

substantive_durs = []

for s in ds:
    segs = s['annotated_transcript']
    total_raw_segments += len(segs)

    flags = []
    for seg in segs:
        dur = seg['end_time'] - seg['start_time']
        text = seg['transcript'].strip()
        if dur <= 0:
            invalid_duration_segments += 1
            flags.append(False)
        elif text in NOISE_TAGS:
            noise_tag_segments += 1
            flags.append(False)
        elif dur < MIN_SUBSTANTIVE_DURATION:
            short_backchannel_segments += 1
            flags.append(False)
        else:
            substantive_segments += 1
            substantive_durs.append(dur)
            flags.append(True)

    # walk only substantive segments, looking at the NEXT substantive segment
    substantive_idx = [i for i, f in enumerate(flags) if f]
    for pos in range(len(substantive_idx) - 1):
        i, j = substantive_idx[pos], substantive_idx[pos + 1]
        cur, nxt = segs[i], segs[j]
        total_transitions += 1

        overlap = nxt['start_time'] < cur['end_time']
        same_speaker = cur['speaker_id'] == nxt['speaker_id']

        if overlap:
            overlap_count += 1
        if same_speaker:
            same_speaker_count += 1

        if same_speaker and not overlap:
            same_speaker_clean_count += 1
        elif same_speaker and overlap:
            same_speaker_overlap_count += 1
        elif not same_speaker and not overlap:
            diff_speaker_clean_count += 1
        elif not same_speaker and overlap:
            diff_speaker_overlap_count += 1

print(f"=== Segment filtering ===")
print(f"Total raw segments: {total_raw_segments}")
print(f"  Invalid duration (<=0): {invalid_duration_segments}")
print(f"  Noise tags: {noise_tag_segments}")
print(f"  Short backchannel (<{MIN_SUBSTANTIVE_DURATION}s): {short_backchannel_segments}")
print(f"  Substantive (usable): {substantive_segments}")

print(f"\n=== Transitions between consecutive substantive segments ===")
print(f"Total: {total_transitions}")
print(f"  Overall overlap rate: {100*overlap_count/total_transitions:.1f}%")
print(f"  Overall same-speaker rate: {100*same_speaker_count/total_transitions:.1f}%")

print(f"\n=== Usable example breakdown ===")
print(f"  same-speaker, clean gap -> label FALSE:     {same_speaker_clean_count} "
      f"({100*same_speaker_clean_count/total_transitions:.1f}%)")
print(f"  same-speaker, overlap   -> label FALSE:     {same_speaker_overlap_count} "
      f"({100*same_speaker_overlap_count/total_transitions:.1f}%)")
print(f"  diff-speaker, clean gap -> label TRUE:       {diff_speaker_clean_count} "
      f"({100*diff_speaker_clean_count/total_transitions:.1f}%)")
print(f"  diff-speaker, overlap   -> EXCLUDED (ambiguous): {diff_speaker_overlap_count} "
      f"({100*diff_speaker_overlap_count/total_transitions:.1f}%)")

total_usable = same_speaker_clean_count + same_speaker_overlap_count + diff_speaker_clean_count
print(f"\nTotal usable cross-segment examples: {total_usable}")
print(f"  -> {same_speaker_clean_count + same_speaker_overlap_count} FALSE (incomplete)")
print(f"  -> {diff_speaker_clean_count} TRUE (complete)")

long_substantive = sum(1 for d in substantive_durs if d > 8)
print(f"\nSubstantive segment durations: min={min(substantive_durs):.2f} "
      f"max={max(substantive_durs):.2f} avg={sum(substantive_durs)/len(substantive_durs):.2f}")
print(f"Substantive segments >8s (extra mid-cut negatives available): {long_substantive}")

In [ ]:
from datasets import load_dataset
import glob, os

NOISE_TAGS = ("<unintelligible>", "<vocalization>", "<laughter>")
MIN_SUBSTANTIVE_DURATION = 1.0
CLIP_SECONDS = 8.0
MIN_CLIP_DURATION = 3.0

DATA_ROOT = "data_diarbench_raw"


def get_languages():
    return sorted([d for d in os.listdir(DATA_ROOT) if os.path.isdir(f"{DATA_ROOT}/{d}") and d != ".cache"])


def is_substantive(seg):
    dur = seg["end_time"] - seg["start_time"]
    text = seg["transcript"].strip()
    return dur > 0 and text not in NOISE_TAGS and dur >= MIN_SUBSTANTIVE_DURATION


def window_is_clean(segs, window_start, window_end, speaker_id, exclude_idx):
    """True if no OTHER segment (any speaker, any type, including noise
    tags) intersects [window_start, window_end]."""
    for k, seg in enumerate(segs):
        if k == exclude_idx:
            continue
        if seg["speaker_id"] == speaker_id:
            continue
        if seg["start_time"] < window_end and seg["end_time"] > window_start:
            return False
    return True


def analyze_language(lang):
    files = sorted(glob.glob(f"{DATA_ROOT}/{lang}/**/*.parquet", recursive=True))
    if not files:
        return None
    ds = load_dataset("parquet", data_files=files, split="train")

    usable_true = 0
    usable_false_cross = 0
    usable_false_midcut = 0
    rejected_dirty_window = 0
    rejected_short_clip = 0

    for sample in ds:
        segs = sample["annotated_transcript"]
        flags = [is_substantive(s) for s in segs]
        idx = [i for i, f in enumerate(flags) if f]

        for pos in range(len(idx) - 1):
            i, j = idx[pos], idx[pos + 1]
            cur, nxt = segs[i], segs[j]
            same_speaker = cur["speaker_id"] == nxt["speaker_id"]

            end_time = cur["end_time"]
            window_start = max(0.0, end_time - CLIP_SECONDS)
            actual_dur = end_time - window_start
            if actual_dur < MIN_CLIP_DURATION:
                rejected_short_clip += 1
                continue

            if not window_is_clean(segs, window_start, end_time, cur["speaker_id"], i):
                rejected_dirty_window += 1
                continue

            if same_speaker:
                usable_false_cross += 1
            else:
                usable_true += 1

        for i in idx:
            seg = segs[i]
            dur = seg["end_time"] - seg["start_time"]
            if dur > CLIP_SECONDS:
                mid_time = seg["start_time"] + dur * 0.5
                window_start = max(0.0, mid_time - CLIP_SECONDS)
                if window_is_clean(segs, window_start, mid_time, seg["speaker_id"], i):
                    usable_false_midcut += 1
                else:
                    rejected_dirty_window += 1

    return {
        "lang": lang, "usable_true": usable_true,
        "usable_false_cross": usable_false_cross, "usable_false_midcut": usable_false_midcut,
        "rejected_dirty_window": rejected_dirty_window, "rejected_short_clip": rejected_short_clip,
    }


results = []
for lang in get_languages():
    print(f"Processing {lang}...")
    r = analyze_language(lang)
    if r:
        results.append(r)

print(f"\n{'Language':<12} {'TRUE':>8} {'FALSE(cross)':>13} {'FALSE(midcut)':>14} "
      f"{'rej_dirty':>10} {'rej_short':>10}")
for r in results:
    print(f"{r['lang']:<12} {r['usable_true']:>8} {r['usable_false_cross']:>13} "
          f"{r['usable_false_midcut']:>14} {r['rejected_dirty_window']:>10} {r['rejected_short_clip']:>10}")

tot_true = sum(r['usable_true'] for r in results)
tot_false_cross = sum(r['usable_false_cross'] for r in results)
tot_false_midcut = sum(r['usable_false_midcut'] for r in results)
print(f"\nTOTAL: {tot_true} TRUE, {tot_false_cross + tot_false_midcut} FALSE "
      f"({tot_false_cross} cross + {tot_false_midcut} midcut)")
print(f"Grand total usable (clean-window) examples: {tot_true + tot_false_cross + tot_false_midcut}")

In [ ]:
import glob, random, os
import soundfile as sf
from datasets import load_dataset

NOISE_TAGS = ("<unintelligible>", "<vocalization>", "<laughter>")
MIN_SUBSTANTIVE_DURATION = 1.0
CLIP_SECONDS = 8.0
MIN_CLIP_DURATION = 3.0

random.seed(3)

files = sorted(glob.glob("data_diarbench_raw/Hindi/**/*.parquet", recursive=True))
ds = load_dataset("parquet", data_files=files, split="train")

def is_substantive(seg):
    dur = seg["end_time"] - seg["start_time"]
    text = seg["transcript"].strip()
    return dur > 0 and text not in NOISE_TAGS and dur >= MIN_SUBSTANTIVE_DURATION

def window_is_clean(segs, window_start, window_end, speaker_id, exclude_idx):
    for k, seg in enumerate(segs):
        if k == exclude_idx:
            continue
        if seg["speaker_id"] == speaker_id:
            continue
        if seg["start_time"] < window_end and seg["end_time"] > window_start:
            return False
    return True

candidates = []
for row_idx in range(len(ds)):
    sample = ds[row_idx]
    segs = sample["annotated_transcript"]
    flags = [is_substantive(s) for s in segs]
    idx = [i for i, f in enumerate(flags) if f]

    for pos in range(len(idx) - 1):
        i, j = idx[pos], idx[pos + 1]
        cur, nxt = segs[i], segs[j]
        same_speaker = cur["speaker_id"] == nxt["speaker_id"]

        end_time = cur["end_time"]
        window_start = max(0.0, end_time - CLIP_SECONDS)
        if end_time - window_start < MIN_CLIP_DURATION:
            continue
        if not window_is_clean(segs, window_start, end_time, cur["speaker_id"], i):
            continue

        label = False if same_speaker else True
        candidates.append({
            "row_idx": row_idx, "end_time": end_time, "window_start": window_start,
            "label": label, "kind": "cross_segment", "transcript": cur["transcript"],
        })

    for i in idx:
        seg = segs[i]
        dur = seg["end_time"] - seg["start_time"]
        if dur > CLIP_SECONDS:
            mid_time = seg["start_time"] + dur * 0.5
            window_start = max(0.0, mid_time - CLIP_SECONDS)
            if window_is_clean(segs, window_start, mid_time, seg["speaker_id"], i):
                candidates.append({
                    "row_idx": row_idx, "end_time": mid_time, "window_start": window_start,
                    "label": False, "kind": "mid_cut", "transcript": seg["transcript"],
                })

    if len(candidates) > 300:
        break

true_ex = [c for c in candidates if c["label"] and c["kind"] == "cross_segment"]
false_ex = [c for c in candidates if not c["label"] and c["kind"] == "cross_segment"]
midcut_ex = [c for c in candidates if c["kind"] == "mid_cut"]
print(f"True: {len(true_ex)}, False(cross): {len(false_ex)}, mid_cut: {len(midcut_ex)}")

sample_set = (random.sample(true_ex, min(6, len(true_ex)))
              + random.sample(false_ex, min(6, len(false_ex)))
              + random.sample(midcut_ex, min(6, len(midcut_ex))))

os.makedirs("hindi_clean_sample", exist_ok=True)
for i, ex in enumerate(sample_set):
    row = ds[ex["row_idx"]]
    clip = row["audio"].get_samples_played_in_range(ex["window_start"], ex["end_time"])
    waveform = clip.data.squeeze(0).numpy()

    fname = f"hindi_clean_sample/{i:02d}_label{ex['label']}_{ex['kind']}.wav"
    sf.write(fname, waveform, clip.sample_rate)
    print(fname, f"| end={ex['end_time']:.2f}s dur={ex['end_time']-ex['window_start']:.2f}s "
                 f"| tail: ...{ex['transcript'][-50:]}")

print("\nDownload hindi_clean_sample/ and listen.")

In [ ]:
import glob, re
from datasets import load_dataset
from collections import Counter

DATA_ROOT = "data_diarbench_raw"
import os

def get_languages():
    return sorted([d for d in os.listdir(DATA_ROOT) if os.path.isdir(f"{DATA_ROOT}/{d}") and d != ".cache"])

tag_pattern = re.compile(r"<[^>]+>")  # matches anything like <word> or <multi word>

all_tags = Counter()
tag_examples = {}  # tag -> one example full transcript for context
transcript_lengths_by_tag = {}  # tag -> list of durations when transcript IS exactly that tag

for lang in get_languages():
    files = sorted(glob.glob(f"{DATA_ROOT}/{lang}/**/*.parquet", recursive=True))
    if not files:
        continue
    ds = load_dataset("parquet", data_files=files, split="train")

    for sample in ds:
        for seg in sample["annotated_transcript"]:
            text = seg["transcript"].strip()
            found = tag_pattern.findall(text)
            for tag in found:
                all_tags[tag] += 1
                if tag not in tag_examples:
                    tag_examples[tag] = text[:100]

            # separately: track duration distribution when the WHOLE transcript is just one tag
            if found and text == found[0]:
                dur = seg["end_time"] - seg["start_time"]
                transcript_lengths_by_tag.setdefault(found[0], []).append(dur)

    print(f"Done: {lang}")

print(f"\n=== All distinct tags found ({len(all_tags)}) ===")
for tag, count in all_tags.most_common():
    print(f"  {tag}: {count} occurrences | example: {tag_examples[tag]}")

print(f"\n=== Duration stats for segments that are ONLY a tag (nothing else) ===")
for tag, durs in transcript_lengths_by_tag.items():
    print(f"  {tag}: n={len(durs)} min={min(durs):.2f}s max={max(durs):.2f}s avg={sum(durs)/len(durs):.2f}s "
          f"| >5s: {sum(1 for d in durs if d>5)} | >10s: {sum(1 for d in durs if d>10)}")

In [ ]:
import glob, os, re
from datasets import load_dataset

BRACKET_PATTERN = re.compile(r"<[^>]*>|\([^)]*\)")
MIN_SUBSTANTIVE_DURATION = 1.0
CLIP_SECONDS = 8.0
MIN_CLIP_DURATION = 3.0

DATA_ROOT = "data_diarbench_raw"

def get_languages():
    return sorted([d for d in os.listdir(DATA_ROOT) if os.path.isdir(f"{DATA_ROOT}/{d}") and d != ".cache"])

def strip_tags(text):
    return BRACKET_PATTERN.sub("", text).strip()

def is_substantive(seg):
    dur = seg["end_time"] - seg["start_time"]
    if dur <= 0 or dur < MIN_SUBSTANTIVE_DURATION:
        return False
    real_text = strip_tags(seg["transcript"])
    return len(real_text) > 0

def has_real_speech(seg):
    """Used for window contamination check - True if segment has ANY real
    content after stripping tags, regardless of duration threshold."""
    return len(strip_tags(seg["transcript"])) > 0

def window_is_clean(segs, window_start, window_end, speaker_id, exclude_idx):
    for k, seg in enumerate(segs):
        if k == exclude_idx:
            continue
        if seg["speaker_id"] == speaker_id:
            continue
        if not has_real_speech(seg):
            continue  # pure noise/tag segment - doesn't count as contamination
        if seg["start_time"] < window_end and seg["end_time"] > window_start:
            return False
    return True

def analyze_language(lang):
    files = sorted(glob.glob(f"{DATA_ROOT}/{lang}/**/*.parquet", recursive=True))
    if not files:
        return None
    ds = load_dataset("parquet", data_files=files, split="train")

    usable_true = 0
    usable_false_cross = 0
    usable_false_midcut = 0
    rejected_dirty_window = 0
    rejected_short_clip = 0

    for sample in ds:
        segs = sample["annotated_transcript"]
        flags = [is_substantive(s) for s in segs]
        idx = [i for i, f in enumerate(flags) if f]

        for pos in range(len(idx) - 1):
            i, j = idx[pos], idx[pos + 1]
            cur, nxt = segs[i], segs[j]
            same_speaker = cur["speaker_id"] == nxt["speaker_id"]

            end_time = cur["end_time"]
            window_start = max(0.0, end_time - CLIP_SECONDS)
            if end_time - window_start < MIN_CLIP_DURATION:
                rejected_short_clip += 1
                continue

            if not window_is_clean(segs, window_start, end_time, cur["speaker_id"], i):
                rejected_dirty_window += 1
                continue

            if same_speaker:
                usable_false_cross += 1
            else:
                usable_true += 1

        for i in idx:
            seg = segs[i]
            dur = seg["end_time"] - seg["start_time"]
            if dur > CLIP_SECONDS:
                mid_time = seg["start_time"] + dur * 0.5
                window_start = max(0.0, mid_time - CLIP_SECONDS)
                if window_is_clean(segs, window_start, mid_time, seg["speaker_id"], i):
                    usable_false_midcut += 1
                else:
                    rejected_dirty_window += 1

    return {
        "lang": lang, "usable_true": usable_true,
        "usable_false_cross": usable_false_cross, "usable_false_midcut": usable_false_midcut,
        "rejected_dirty_window": rejected_dirty_window, "rejected_short_clip": rejected_short_clip,
    }

results = []
for lang in get_languages():
    print(f"Processing {lang}...")
    r = analyze_language(lang)
    if r:
        results.append(r)

print(f"\n{'Language':<12} {'TRUE':>8} {'FALSE(cross)':>13} {'FALSE(midcut)':>14} "
      f"{'rej_dirty':>10} {'rej_short':>10}")
for r in results:
    print(f"{r['lang']:<12} {r['usable_true']:>8} {r['usable_false_cross']:>13} "
          f"{r['usable_false_midcut']:>14} {r['rejected_dirty_window']:>10} {r['rejected_short_clip']:>10}")

tot_true = sum(r['usable_true'] for r in results)
tot_false_cross = sum(r['usable_false_cross'] for r in results)
tot_false_midcut = sum(r['usable_false_midcut'] for r in results)
print(f"\nTOTAL: {tot_true} TRUE, {tot_false_cross + tot_false_midcut} FALSE "
      f"({tot_false_cross} cross + {tot_false_midcut} midcut)")
print(f"Grand total usable examples: {tot_true + tot_false_cross + tot_false_midcut}")

In [ ]:
import glob, random, os
import soundfile as sf
from datasets import load_dataset

random.seed(4)

files = sorted(glob.glob("data_diarbench_raw/Hindi/**/*.parquet", recursive=True))
ds = load_dataset("parquet", data_files=files, split="train")

candidates = []
for row_idx in range(len(ds)):
    sample = ds[row_idx]
    segs = sample["annotated_transcript"]
    flags = [is_substantive(s) for s in segs]
    idx = [i for i, f in enumerate(flags) if f]

    for pos in range(len(idx) - 1):
        i, j = idx[pos], idx[pos + 1]
        cur, nxt = segs[i], segs[j]
        same_speaker = cur["speaker_id"] == nxt["speaker_id"]

        end_time = cur["end_time"]
        window_start = max(0.0, end_time - CLIP_SECONDS)
        if end_time - window_start < MIN_CLIP_DURATION:
            continue
        if not window_is_clean(segs, window_start, end_time, cur["speaker_id"], i):
            continue

        label = False if same_speaker else True
        candidates.append({
            "row_idx": row_idx, "end_time": end_time, "window_start": window_start,
            "label": label, "kind": "cross_segment", "transcript": cur["transcript"],
        })

    for i in idx:
        seg = segs[i]
        dur = seg["end_time"] - seg["start_time"]
        if dur > CLIP_SECONDS:
            mid_time = seg["start_time"] + dur * 0.5
            window_start = max(0.0, mid_time - CLIP_SECONDS)
            if window_is_clean(segs, window_start, mid_time, seg["speaker_id"], i):
                candidates.append({
                    "row_idx": row_idx, "end_time": mid_time, "window_start": window_start,
                    "label": False, "kind": "mid_cut", "transcript": seg["transcript"],
                })

    if len(candidates) > 300:
        break

true_ex = [c for c in candidates if c["label"] and c["kind"] == "cross_segment"]
false_ex = [c for c in candidates if not c["label"] and c["kind"] == "cross_segment"]
midcut_ex = [c for c in candidates if c["kind"] == "mid_cut"]
print(f"True: {len(true_ex)}, False(cross): {len(false_ex)}, mid_cut: {len(midcut_ex)}")

sample_set = (random.sample(true_ex, min(6, len(true_ex)))
              + random.sample(false_ex, min(6, len(false_ex)))
              + random.sample(midcut_ex, min(6, len(midcut_ex))))

os.makedirs("hindi_v2_sample", exist_ok=True)
for i, ex in enumerate(sample_set):
    row = ds[ex["row_idx"]]
    clip = row["audio"].get_samples_played_in_range(ex["window_start"], ex["end_time"])
    waveform = clip.data.squeeze(0).numpy()

    fname = f"hindi_v2_sample/{i:02d}_label{ex['label']}_{ex['kind']}.wav"
    sf.write(fname, waveform, clip.sample_rate)
    print(fname, f"| dur={ex['end_time']-ex['window_start']:.2f}s | tail: ...{ex['transcript'][-60:]}")

print("\nDownload hindi_v2_sample/ and listen.")

In [ ]:
import os
import torch
from safetensors.torch import load_file
from transformers import WhisperConfig, Trainer, TrainingArguments, WhisperFeatureExtractor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from model import SmartTurnV3Model
from dataloader import CHUNK_LENGTH_SECONDS
from dataloader_stage2 import load_stage2_datasets, Stage2Collator


def load_finetuned_model(checkpoint_dir: str) -> SmartTurnV3Model:
    config = WhisperConfig.from_pretrained(checkpoint_dir)
    model = SmartTurnV3Model(config)

    safetensors_path = os.path.join(checkpoint_dir, "model.safetensors")
    bin_path = os.path.join(checkpoint_dir, "pytorch_model.bin")

    if os.path.exists(safetensors_path):
        state_dict = load_file(safetensors_path)
    elif os.path.exists(bin_path):
        state_dict = torch.load(bin_path, map_location="cpu")
    else:
        raise FileNotFoundError(f"No weights file found in {checkpoint_dir}")

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f"Loaded checkpoint - missing: {missing}, unexpected: {unexpected}")
    return model


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = logits.squeeze()
    preds = (probs > 0.5).astype(int)
    labels = labels.astype(int)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
    }


def slice_accuracy(trainer, dataset, get_key_fn, label_name):
    predictions = trainer.predict(dataset)
    probs = predictions.predictions.squeeze()
    preds = (probs > 0.5).astype(int)
    labels = predictions.label_ids.astype(int)

    slice_stats = {}
    for idx in range(len(dataset)):
        item = dataset[idx]
        key = get_key_fn(item)
        slice_stats.setdefault(key, [0, 0])
        slice_stats[key][0] += int(preds[idx] == labels[idx])
        slice_stats[key][1] += 1

    print(f"\n=== Accuracy by {label_name} ===")
    for k, (correct, total) in sorted(slice_stats.items(), key=lambda x: -x[1][0]/x[1][1]):
        print(f"    {k:>15}: {correct/total:.4f}  (n={total})")


def main(
    stage1_checkpoint: str = "checkpoints/final_model",
    stage2_manifest: str = "data_stage2_materialized/manifest.csv",
    stage2_data_root: str = "data_stage2_materialized",
    batch_size: int = 16,
):
    print(f"Loading stage-1 checkpoint from {stage1_checkpoint}...")
    model = load_finetuned_model(stage1_checkpoint)

    fe = WhisperFeatureExtractor(chunk_length=CHUNK_LENGTH_SECONDS)
    collate_fn = Stage2Collator(fe)

    print("Loading FULL stage-2 dataset (zero-shot, no training)...")
    train_ds, eval_ds = load_stage2_datasets(stage2_manifest, stage2_data_root, eval_fraction=1.0)

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir="/tmp/zero_shot_eval",
            per_device_eval_batch_size=batch_size,
            report_to=[],
            remove_unused_columns=False,
            fp16=True,
        ),
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    print("\nRunning zero-shot evaluation on full stage-2 dataset...")
    metrics = trainer.evaluate(eval_ds)
    print(f"\n=== Overall zero-shot metrics ===")
    for k, v in metrics.items():
        print(f"  {k}: {v}")

    slice_accuracy(trainer, eval_ds, lambda item: item["language"], "language")
    slice_accuracy(trainer, eval_ds, lambda item: item["kind"], "kind")


main()

In [ ]:
from datasets import load_dataset
from transformers import Trainer, TrainingArguments, WhisperFeatureExtractor, WhisperConfig
from safetensors.torch import load_file
from dataloader import WhisperCollator, CHUNK_LENGTH_SECONDS
from model import SmartTurnV3Model
import time

print('Loading small test slice...')
ds = load_dataset('parquet', data_files='data_full_raw/test/data/*.parquet', split='train')
small_ds = ds.select(range(100))
print(f'Testing on {len(small_ds)} rows')

print('Loading model (manual state_dict load, NOT from_pretrained)...')
config = WhisperConfig.from_pretrained('checkpoints/final_model')
model = SmartTurnV3Model(config)
sd = load_file('checkpoints/final_model/model.safetensors')
missing, unexpected = model.load_state_dict(sd, strict=False)
print(f'missing: {missing}, unexpected: {unexpected}')

fe = WhisperFeatureExtractor(chunk_length=CHUNK_LENGTH_SECONDS)
collate_fn = WhisperCollator(fe)

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir='/tmp/test_eval',
        per_device_eval_batch_size=16,
        report_to=[],
        remove_unused_columns=False,
        fp16=True,
        dataloader_num_workers=8,
    ),
    data_collator=collate_fn,
)

t0 = time.time()
result = trainer.predict(small_ds)
elapsed = time.time() - t0
print(f'{len(small_ds)} rows took {elapsed:.1f}s ({len(small_ds)/elapsed:.2f} rows/sec)')

In [ ]:
import sys
print(sys.version)